### ChromaDB 설치

In [ ]:
pip install chromadb

In [ ]:
# `sentence_transformers` 필요 시 설치
#pip install sentence_transformers

### 통합할 ChromaDB 업로드

**✅ 반드시 해당 Chroma DB는 모든 파일이 온전한 상태로 있어야 DB 병합이 정상적으로 이루어짐**

In [3]:
from chromadb import PersistentClient

db1 = PersistentClient(path="/content/drive/MyDrive/Colab Notebooks/chroma_data/chroma_db_nic")
db2 = PersistentClient(path="/content/drive/MyDrive/Colab Notebooks/chroma_data/chroma_db_post")
db3 = PersistentClient(path="/content/drive/MyDrive/Colab Notebooks/chroma_data/chroma_db_place")

#### 컬렉션 확인

In [4]:
print("📂 DB1에 있는 컬렉션:", [c.name for c in db1.list_collections()])
print("📂 DB2에 있는 컬렉션:", [c.name for c in db2.list_collections()])
print("📂 DB3에 있는 컬렉션:", [c.name for c in db3.list_collections()])

📂 DB1에 있는 컬렉션: ['hobby_subtraits', 'mbti_traits']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📂 DB2에 있는 컬렉션: ['mbti_feeds', 'user_mbti_latest', 'user_mbti_history', 'user_hobby_latest', 'user_hobby_history']
📂 DB3에 있는 컬렉션: ['review_collection', 'menu_collection', 'recommendation_history']


#### 컬렉션 이름 리스트화

In [5]:
c1, c2, c3 = [c.name for c in db1.list_collections()], [c.name for c in db2.list_collections()], [c.name for c in db3.list_collections()]

In [8]:
merged_db = PersistentClient(path="./chroma_db_merged")

# 복사할 컬렉션 이름들
collections_to_copy = []
arr = [c1, c2, c3]
db_list = [db1, db2, db3]

for i in range(len(arr)):
    for arr_collection in arr[i]:
        collections_to_copy.append((arr_collection, db_list[i]))

In [14]:
collections_to_copy

[('hobby_subtraits', <chromadb.api.client.Client at 0x7e7deede2c10>),
 ('mbti_traits', <chromadb.api.client.Client at 0x7e7deede2c10>),
 ('mbti_feeds', <chromadb.api.client.Client at 0x7e7dedb34a50>),
 ('user_mbti_latest', <chromadb.api.client.Client at 0x7e7dedb34a50>),
 ('user_mbti_history', <chromadb.api.client.Client at 0x7e7dedb34a50>),
 ('user_hobby_latest', <chromadb.api.client.Client at 0x7e7dedb34a50>),
 ('user_hobby_history', <chromadb.api.client.Client at 0x7e7dedb34a50>),
 ('review_collection', <chromadb.api.client.Client at 0x7e7dedb57950>),
 ('menu_collection', <chromadb.api.client.Client at 0x7e7dedb57950>),
 ('recommendation_history', <chromadb.api.client.Client at 0x7e7dedb57950>)]

### 배치 단위로 통합 Chroma DB에 업로드

In [15]:
BATCH_SIZE = 4000

for col_name, src_db in collections_to_copy:
    print(f"📦 복사 중: {col_name}")

    src_col = src_db.get_collection(col_name)
    dest_col = merged_db.get_or_create_collection(name=col_name)

    offset = 0
    total_copied = 0

    while True:
        data = src_col.get(
            limit=BATCH_SIZE,
            offset=offset,
            include=["embeddings", "documents", "metadatas"]
        )

        if not data["ids"]:
            break  # ✅ 더 이상 불러올 데이터 없음 → 루프 종료

        n = len(data["ids"])
        total_copied += n

        # ID 중복 방지를 위해 접두사 추가
        prefixed_ids = [f"{col_name}__{id}" for id in data["ids"]]

        if not (n == len(data["embeddings"]) == len(data["documents"]) == len(data["metadatas"])):
            raise ValueError(
                f"❌ 길이 불일치 발생 in {col_name}:\n"
                f"    ids({n}), emb({len(data['embeddings'])}), "
                f"doc({len(data['documents'])}), meta({len(data['metadatas'])})"
            )

        dest_col.add(
            ids=prefixed_ids,
            embeddings=data["embeddings"],
            documents=data["documents"],
            metadatas=data["metadatas"]
        )

        print(f"  └─ ✅ {offset}~{offset + n}번 항목 복사 완료")

        offset += n  # 다음 배치를 위해 오프셋 증가

    print(f"🎉 {col_name} 전체 복사 완료 → 총 {total_copied}개")
    print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

📦 복사 중: hobby_subtraits
  └─ ✅ 0~4000번 항목 복사 완료
  └─ ✅ 4000~6000번 항목 복사 완료
🎉 hobby_subtraits 전체 복사 완료 → 총 6000개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: mbti_traits
  └─ ✅ 0~4000번 항목 복사 완료
  └─ ✅ 4000~6528번 항목 복사 완료
🎉 mbti_traits 전체 복사 완료 → 총 6528개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: mbti_feeds
  └─ ✅ 0~4000번 항목 복사 완료
  └─ ✅ 4000~8000번 항목 복사 완료
  └─ ✅ 8000~12000번 항목 복사 완료
  └─ ✅ 12000~16000번 항목 복사 완료
  └─ ✅ 16000~20000번 항목 복사 완료
  └─ ✅ 20000~24000번 항목 복사 완료
  └─ ✅ 24000~28000번 항목 복사 완료
  └─ ✅ 28000~31069번 항목 복사 완료
🎉 mbti_feeds 전체 복사 완료 → 총 31069개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: user_mbti_latest
  └─ ✅ 0~3번 항목 복사 완료
🎉 user_mbti_latest 전체 복사 완료 → 총 3개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: user_mbti_history
  └─ ✅ 0~30번 항목 복사 완료
🎉 user_mbti_history 전체 복사 완료 → 총 30개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: user_hobby_latest
  └─ ✅ 0~3번 항목 복사 완료
🎉 user_hobby_latest 전체 복사 완료 → 총 3개
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 복사 중: user_hobby_histor

### 통합 완료 여부 테스트

In [13]:
print("\n🔍 복사 완료 후 컬렉션 검증 시작")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

for col_name, _ in collections_to_copy:
    dest_col = merged_db.get_collection(col_name)
    count = dest_col.count()
    print(f"📁 {col_name} → 총 {count}개 저장됨 ✅")

    # 일부 샘플 확인 (선택)
    sample = dest_col.get(limit=1, include=["documents", "metadatas"])
    try:
        print(f"  └ 샘플 ID: {sample['ids'][0]}")
        print(f"  └ 샘플 문서 요약: {str(sample['documents'][0])[:60]}...")
        print(f"  └ 샘플 메타데이터: {sample['metadatas'][0]}")
    except:
        print('데이터 없음')


🔍 복사 완료 후 컬렉션 검증 시작
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📁 hobby_subtraits → 총 6000개 저장됨 ✅
  └ 샘플 ID: hobby_subtraits__a85c0d60-42e1-441e-939c-c63fee139f3e
  └ 샘플 문서 요약: 드리블...
  └ 샘플 메타데이터: {'Weight': 0.9, 'Hobby': '축구(하기)', 'Subtrait': '드리블'}
📁 mbti_traits → 총 6528개 저장됨 ✅
  └ 샘플 ID: mbti_traits__933a5c7c-3750-464a-8820-13de29e48eca
  └ 샘플 문서 요약: 창의적...
  └ 샘플 메타데이터: {'Trait': '창의적', 'MBTI': 'ENTP', 'Weight': 1.0}
📁 mbti_feeds → 총 31069개 저장됨 ✅
  └ 샘플 ID: mbti_feeds__id_0
  └ 샘플 문서 요약: 공방 가서 맞추는 건 싫고 금으로 맞추는 거 아님[SEP]싫어요 ㅎ[SEP]...
  └ 샘플 메타데이터: {'a_mbti': 'istj'}
📁 user_mbti_latest → 총 3개 저장됨 ✅
  └ 샘플 ID: user_mbti_latest__1
  └ 샘플 문서 요약: ...
  └ 샘플 메타데이터: {'updated_at': '2025-05-08T09:03:30.721907', 'hobby': '등산', 'user_id': '1'}
📁 user_mbti_history → 총 30개 저장됨 ✅
  └ 샘플 ID: user_mbti_history__bf947fb8-9148-484b-9710-952833b3453d
  └ 샘플 문서 요약: ...
  └ 샘플 메타데이터: {'hobby': '', 'timestamp': '2025-05-06T09:15:27.794252', 'user_id': 'test_user'}
📁 user_hobby_latest → 총 3개 저장됨 ✅
  └ 샘플 ID